In [1]:
import os
import numpy as np
import pandas as pd
import pickle
from pandas import HDFStore

In [2]:
ASSETS = {'KGHM': [10783], 'PKNORLEN': [11319], 'PKOBP': [11314], 'PZU': [10735], 'PEKAO': [11322]}
data_path = r'../msg_data/trades/' 

### Available keys in hdf5 files
Example for KGHM but for all files keys are the same

In [3]:
asset = 'KGHM'
with HDFStore(os.path.join(data_path, f'{asset}_trades_2017_zlib.h5'), mode='r') as hdf:
    keys = hdf.keys()
keys

['/d20170102',
 '/d20170103',
 '/d20170104',
 '/d20170105',
 '/d20170109',
 '/d20170110',
 '/d20170111',
 '/d20170112',
 '/d20170113',
 '/d20170116',
 '/d20170117',
 '/d20170118',
 '/d20170119',
 '/d20170120',
 '/d20170123',
 '/d20170124',
 '/d20170125',
 '/d20170126',
 '/d20170127',
 '/d20170130',
 '/d20170131',
 '/d20170201',
 '/d20170202',
 '/d20170203',
 '/d20170206',
 '/d20170207',
 '/d20170208',
 '/d20170209',
 '/d20170210',
 '/d20170213',
 '/d20170214',
 '/d20170215',
 '/d20170216',
 '/d20170217',
 '/d20170220',
 '/d20170221',
 '/d20170222',
 '/d20170223',
 '/d20170224',
 '/d20170227',
 '/d20170228',
 '/d20170301',
 '/d20170302',
 '/d20170303',
 '/d20170306',
 '/d20170307',
 '/d20170308',
 '/d20170309',
 '/d20170310',
 '/d20170313',
 '/d20170314',
 '/d20170315',
 '/d20170316',
 '/d20170317',
 '/d20170320',
 '/d20170321',
 '/d20170322',
 '/d20170323',
 '/d20170324',
 '/d20170327',
 '/d20170328',
 '/d20170329',
 '/d20170330',
 '/d20170331',
 '/d20170403',
 '/d20170404',
 '/d201704

### Example of DataFrame

In [4]:
asset = 'KGHM'
key = '/d20170110'
df = pd.read_hdf(os.path.join(data_path, f'{asset}_trades_2017_zlib.h5'), key)
df.head()

,time,symbol_idx,trade_id,trading_datetime,price,volume,opening_trade_indicator,trade_origin,price_level
245136,1484035200006035000,10783,1,1484035200006035000,9690,5,O,B,2
245137,1484035200006128000,10783,2,1484035200006128000,9690,5,O,B,2
245138,1484035200006133000,10783,3,1484035200006133000,9690,10,O,B,2
245139,1484035200006135000,10783,4,1484035200006135000,9690,30,O,B,2
245140,1484035200006137000,10783,5,1484035200006137000,9690,10,O,B,2


### Description of columns
Columns `symbol_idx`+`trade_id`+`trading_datetime` uniquely identify transactions.

- `time` - int - Time of sending the message in the system, timestamps
- `symbol_idx` - int - UTP-MD proprietary identification of the instrument
- `trade_id` - int - Unique numeric and increasing Identifier of the trade, set by the trading engine
- `trading_datetime` - int - Trade generation time, timestamps
- `price` - int - Price of the trade (To be calculated with the `price_level`)
- `volume` - int - Number of securities transacted in this
- `opening_trade_indicator` - str - If the trade took place during the opening auction or during the core session. Valid values:
   - `O` - Opening
   - `S` - Core Continuose
   - `NaN` - Missing
- `trade_origin` - str - This field indicates the origin of the trade. Valid values:
   - `B` - Orders from the book
   - `O` - Block trade
   - `I` - IPO trade
   - `T` - Tender offer trade
   - `S` - BISO trade
   - `R` - Trade reporting (for future use)
   - `NaN` - Missing
- `price_level` - int - Divide factor to combine with `price` in order to convert to float, (`price/(10^price_level)`)

### Conversion of  timestamps to datetime and scaling of price

In [5]:
df['time'] = (pd.to_datetime(df['time'], unit='ns', utc=True)).dt.tz_convert('Europe/Warsaw')
df['trading_datetime'] = (pd.to_datetime(df['trading_datetime'], unit='ns', utc=True)).dt.tz_convert('Europe/Warsaw')
df['price_float'] = df['price'] / (10**df['price_level'])
df.head()

,time,symbol_idx,trade_id,trading_datetime,price,volume,opening_trade_indicator,trade_origin,price_level,price_float
245136,2017-01-10 09:00:00.006035+01:00,10783,1,2017-01-10 09:00:00.006035+01:00,9690,5,O,B,2,96.9
245137,2017-01-10 09:00:00.006128+01:00,10783,2,2017-01-10 09:00:00.006128+01:00,9690,5,O,B,2,96.9
245138,2017-01-10 09:00:00.006133+01:00,10783,3,2017-01-10 09:00:00.006133+01:00,9690,10,O,B,2,96.9
245139,2017-01-10 09:00:00.006135+01:00,10783,4,2017-01-10 09:00:00.006135+01:00,9690,30,O,B,2,96.9
245140,2017-01-10 09:00:00.006137+01:00,10783,5,2017-01-10 09:00:00.006137+01:00,9690,10,O,B,2,96.9
